In [3]:

import duckdb
import pandas as pd

con = duckdb.connect("../data/database/insight_agent.duckdb")

In [4]:
query = """
SELECT
    DATE_TRUNC('month', order_purchase_timestamp) AS month,
    COUNT(DISTINCT order_id) AS orders
FROM fact_orders
GROUP BY 1
ORDER BY 1
"""

In [5]:
df = con.execute(query).fetchdf()

df

,month,orders
0,2016-09-01,4
1,2016-10-01,324
2,2016-12-01,1
3,2017-01-01,800
4,2017-02-01,1780
5,2017-03-01,2682
6,2017-04-01,2404
7,2017-05-01,3700
8,2017-06-01,3245
9,2017-07-01,4026


In [1]:
import sys
print(sys.executable)


c:\Users\DELL\Projects\insight-agent\ai_master\Scripts\python.exe


In [6]:
query = """
SELECT
    DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
    ROUND(SUM(i.item_revenue), 2) AS revenue
FROM fact_orders o
JOIN fact_order_items i
    ON o.order_id = i.order_id
GROUP BY 1
ORDER BY 1
"""

revenue_df = con.execute(query).fetchdf()

revenue_df

,month,revenue
0,2016-09-01,354.75
1,2016-10-01,56808.84
2,2016-12-01,19.62
3,2017-01-01,137188.49
4,2017-02-01,286280.62
5,2017-03-01,432048.59
6,2017-04-01,412422.24
7,2017-05-01,586190.95
8,2017-06-01,502963.04
9,2017-07-01,584971.62


In [7]:
query = """
SELECT
    DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
    COUNT(DISTINCT o.order_id) AS orders,
    ROUND(SUM(i.item_revenue), 2) AS revenue,
    ROUND(
        SUM(i.item_revenue) / COUNT(DISTINCT o.order_id),
        2
    ) AS avg_order_value
FROM fact_orders o
JOIN fact_order_items i
    ON o.order_id = i.order_id
GROUP BY 1
ORDER BY 1
"""

monthly_business_df = con.execute(query).fetchdf()

monthly_business_df

,month,orders,revenue,avg_order_value
0,2016-09-01,3,354.75,118.25
1,2016-10-01,308,56808.84,184.44
2,2016-12-01,1,19.62,19.62
3,2017-01-01,789,137188.49,173.88
4,2017-02-01,1733,286280.62,165.19
5,2017-03-01,2641,432048.59,163.59
6,2017-04-01,2391,412422.24,172.49
7,2017-05-01,3660,586190.95,160.16
8,2017-06-01,3217,502963.04,156.35
9,2017-07-01,3969,584971.62,147.39


In [8]:
query = """
WITH monthly AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
        COUNT(DISTINCT o.order_id) AS orders,
        SUM(i.item_revenue) AS revenue,
        SUM(i.item_revenue) / COUNT(DISTINCT o.order_id) AS avg_order_value
    FROM fact_orders o
    JOIN fact_order_items i
        ON o.order_id = i.order_id
    GROUP BY 1
)

SELECT
    month,
    orders,
    ROUND(revenue, 2) AS revenue,
    ROUND(avg_order_value, 2) AS avg_order_value,
    ROUND(
        (revenue / LAG(revenue) OVER (ORDER BY month) - 1) * 100,
        2
    ) AS revenue_growth_pct,
    ROUND(
        (orders / LAG(orders) OVER (ORDER BY month) - 1) * 100,
        2
    ) AS order_growth_pct,
    ROUND(
        (avg_order_value / LAG(avg_order_value) OVER (ORDER BY month) - 1) * 100,
        2
    ) AS aov_growth_pct
FROM monthly
ORDER BY month
"""

growth_df = con.execute(query).fetchdf()

growth_df

,month,orders,revenue,avg_order_value,revenue_growth_pct,order_growth_pct,aov_growth_pct
0,2016-09-01,3,354.75,118.25,NaN,NaN,NaN
1,2016-10-01,308,56808.84,184.44,15913.77,10166.67,55.98
2,2016-12-01,1,19.62,19.62,-99.97,-99.68,-89.36
3,2017-01-01,789,137188.49,173.88,699127.78,78800.00,786.22
4,2017-02-01,1733,286280.62,165.19,108.68,119.65,-4.99
5,2017-03-01,2641,432048.59,163.59,50.92,52.39,-0.97
6,2017-04-01,2391,412422.24,172.49,-4.54,-9.47,5.44
7,2017-05-01,3660,586190.95,160.16,42.13,53.07,-7.15
8,2017-06-01,3217,502963.04,156.35,-14.20,-12.10,-2.38
9,2017-07-01,3969,584971.62,147.39,16.31,23.38,-5.73


In [9]:
query = """
WITH monthly_category AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
        p.product_category_name,
        SUM(i.item_revenue) AS revenue
    FROM fact_orders o
    JOIN fact_order_items i
        ON o.order_id = i.order_id
    JOIN dim_products p
        ON i.product_id = p.product_id
    WHERE o.order_purchase_timestamp >= '2018-07-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY 1, 2
),

comparison AS (
    SELECT
        product_category_name,
        SUM(
            CASE WHEN month = '2018-07-01'
                 THEN revenue ELSE 0 END
        ) AS july_revenue,
        SUM(
            CASE WHEN month = '2018-08-01'
                 THEN revenue ELSE 0 END
        ) AS august_revenue
    FROM monthly_category
    GROUP BY 1
)

SELECT
    product_category_name,
    ROUND(july_revenue, 2) AS july_revenue,
    ROUND(august_revenue, 2) AS august_revenue,
    ROUND(august_revenue - july_revenue, 2) AS revenue_change
FROM comparison
ORDER BY revenue_change ASC
LIMIT 15
"""

category_decline_df = con.execute(query).fetchdf()

category_decline_df

,product_category_name,july_revenue,august_revenue,revenue_change
0,relogios_presentes,106064.72,79391.86,-26672.86
1,cool_stuff,34278.98,19091.64,-15187.34
2,ferramentas_jardim,31122.91,20388.16,-10734.75
3,moveis_escritorio,17388.52,8386.67,-9001.85
4,telefonia_fixa,9658.18,2891.02,-6767.16
5,esporte_lazer,65830.02,60374.75,-5455.27
6,moveis_decoracao,57406.26,51976.65,-5429.61
7,construcao_ferramentas_ferramentas,5779.45,1497.04,-4282.41
8,sinalizacao_e_seguranca,6136.19,2158.67,-3977.52
9,utilidades_domesticas,77967.73,74238.89,-3728.84


In [10]:
query = """
SELECT
    p.product_category_name,
    COUNT(*) FILTER (
        WHERE o.order_purchase_timestamp >= '2018-07-01'
          AND o.order_purchase_timestamp < '2018-08-01'
    ) AS july_items,
    COUNT(*) FILTER (
        WHERE o.order_purchase_timestamp >= '2018-08-01'
          AND o.order_purchase_timestamp < '2018-09-01'
    ) AS august_items,
    ROUND(
        AVG(i.item_revenue) FILTER (
            WHERE o.order_purchase_timestamp >= '2018-07-01'
              AND o.order_purchase_timestamp < '2018-08-01'
        ), 2
    ) AS july_avg_item_value,
    ROUND(
        AVG(i.item_revenue) FILTER (
            WHERE o.order_purchase_timestamp >= '2018-08-01'
              AND o.order_purchase_timestamp < '2018-09-01'
        ), 2
    ) AS august_avg_item_value
FROM fact_orders o
JOIN fact_order_items i
    ON o.order_id = i.order_id
JOIN dim_products p
    ON i.product_id = p.product_id
WHERE o.order_purchase_timestamp >= '2018-07-01'
  AND o.order_purchase_timestamp < '2018-09-01'
GROUP BY 1
ORDER BY
    (
        COUNT(*) FILTER (
            WHERE o.order_purchase_timestamp >= '2018-08-01'
              AND o.order_purchase_timestamp < '2018-09-01'
        )
        -
        COUNT(*) FILTER (
            WHERE o.order_purchase_timestamp >= '2018-07-01'
              AND o.order_purchase_timestamp < '2018-08-01'
        )
    ) ASC
LIMIT 15
"""

category_driver_df = con.execute(query).fetchdf()

category_driver_df

,product_category_name,july_items,august_items,july_avg_item_value,august_avg_item_value
0,relogios_presentes,528,416,200.88,190.85
1,moveis_escritorio,82,30,212.06,279.56
2,ferramentas_jardim,179,138,173.87,147.74
3,eletronicos,159,120,86.87,108.06
4,papelaria,191,170,112.51,111.65
5,eletroportateis,43,25,227.50,299.44
6,NaN,47,32,120.55,91.53
7,moveis_sala,55,40,160.73,167.26
8,eletrodomesticos,79,65,82.81,122.06
9,casa_construcao,61,49,136.04,169.65


In [11]:
query = """
WITH monthly_state AS (
    SELECT
        DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
        c.customer_state,
        SUM(i.item_revenue) AS revenue
    FROM fact_orders o
    JOIN fact_order_items i
        ON o.order_id = i.order_id
    JOIN dim_customers c
        ON o.customer_id = c.customer_id
    WHERE o.order_purchase_timestamp >= '2018-07-01'
      AND o.order_purchase_timestamp < '2018-09-01'
    GROUP BY 1, 2
),

comparison AS (
    SELECT
        customer_state,
        SUM(
            CASE WHEN month = '2018-07-01'
                 THEN revenue ELSE 0 END
        ) AS july_revenue,
        SUM(
            CASE WHEN month = '2018-08-01'
                 THEN revenue ELSE 0 END
        ) AS august_revenue
    FROM monthly_state
    GROUP BY 1
)

SELECT
    customer_state,
    ROUND(july_revenue, 2) AS july_revenue,
    ROUND(august_revenue, 2) AS august_revenue,
    ROUND(august_revenue - july_revenue, 2) AS revenue_change
FROM comparison
ORDER BY revenue_change ASC
LIMIT 10
"""

state_decline_df = con.execute(query).fetchdf()

state_decline_df

,customer_state,july_revenue,august_revenue,revenue_change
0,BA,50805.60,26892.16,-23913.44
1,PE,30325.35,15478.29,-14847.06
2,ES,33688.27,19551.27,-14137.00
3,PB,20455.11,9240.93,-11214.18
4,GO,29374.82,21254.89,-8119.93
5,PR,56408.28,49477.14,-6931.14
6,MT,12645.66,5783.76,-6861.90
7,RJ,120112.84,113529.73,-6583.11
8,CE,17266.74,11881.57,-5385.17
9,PA,15851.64,10965.90,-4885.74
